<div style="text-align: center; padding: 18px 0;">
  <h1 style="color: #16324f; margin-bottom: 8px;">
    Stratégie long-short momentum-value sur les actions Européennes
  </h1>
  <h3 style="color: #5c6770; margin-top: 0;">
    Méthode de notation et de stratégie en Python
  </h3>
  <p style="color: #8a8f98; font-size: 15px;">
    Projet de finance de marché • Python • Analyse quantitative
  </p>
</div>


<h1 style="color: darkgreen;">Enoncé</h2>

#### Règles de construction de l'indice :
L'indice de la stratégie est rééquilibré le dernier jour de chaque mois, selon les règles décrites ci-dessous : 

**Étape 0 : Téléchargement des données**

On a besoin d'un fichier contenant les rendements mensuels et les ratios historiques cours/valeur des actions européennes.
L’objectif de la stratégie consiste à tirer parti à la fois des primes de risque momentum et valeur sur le
marché des actions européennes.

**Étape 1 : Notation des actions**

- Chaque action se voit attribuer un score momentum, calculé comme l'écart-type centré du rendement
moyen mensuel sur les 12 derniers mois à l'exclusion du dernier (pour tenir compte des reversions à court
terme).
- Chaque action se voit attribuer un score valeur, calculé comme l'écart-type centré de l'inverse du ratio
cours/valeur comptable mesuré à la fin du mois précédent.
- Le score global de chaque action est calculé comme la moyenne arithmétique des scores momentum et
valeur.

**Étape 2 : Sélection des actions**

- Le portefeuille long est composé des 15 actions avec les scores globaux les plus élevés.
- Le portefeuille short est composé des 15 actions avec les scores globaux les plus bas.

**Étape 3 : Construction du portefeuille**

- Dans les portefeuilles long et short, les actions sont pondérées de manière proportionnelle aux valeurs
absolues de leurs scores globaux.
- La stratégie investit 100% dans les portefeuilles longs et short respectivement.
En plus du code Python, vous fournirez un fichier Excel contenant :
Les performances mensuelles de la stratégie d'investissement de mars 2008 à mars 2026
Les pondérations mensuelles du portefeuille correspondante.

**Partie 2 Évaluation de la stratégie**

Évaluer la pertinence de la stratégie d'investissement décrite ci-dessus par rapport à des stratégies
long/short aléatoires sur les actions européennes ("lucky").
Détailler la méthodologie et les résultats obtenus.

**Rendu final**

En plus du code Python, vous fournirez un fichier Excel contenant :
Les performances mensuelles de la stratégie d'investissement de mars 2008 à mars 2026
Les pondérations mensuelles du portefeuille correspondante.

<h1 style="color: darkgreen;">Application</h2>

## Étape 0 - Constitution du jeu de données

On prépare ici les deux briques nécessaires pour la suite du projet :

- les rendements mensuels des actions ;
- un ratio historique `price-to-book` reconstruit mensuellement.

Comme Yahoo Finance ne fournit pas directement une série mensuelle historique du ratio cours / valeur comptable, on adopte l'approximation suivante :

- on télécharge les prix de clôture mensuels ;
- on récupère les bilans trimestriels et annuels disponibles ;
- on calcule une valeur comptable par action `book value per share = common equity / shares outstanding` ;
- on propage cette valeur sur les mois suivants jusqu'à la publication suivante.

Cette approximation est cohérente avec une utilisation en backtest mensuel et fournit une base exploitable pour l'étape de scoring valeur.

### Interprétation théorique des fichiers CSV

L'objectif de l'étape 0 est de construire, pour chaque action $i$ et chaque fin de mois $t$, un petit jeu de variables quantitatives qui servira ensuite au scoring. Les fichiers `.csv` générés correspondent chacun à une quantité financière bien définie.

#### 1. Rendements mensuels : `rendements_mensuels_europe.csv`

On note $P_{i,t}$ le prix de clôture ajusté de l'action $i$ à la fin du mois $t$. Le rendement mensuel simple est alors défini par :

$$
R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

Ce rendement mesure la performance de l'action entre la fin du mois $t-1$ et la fin du mois $t$.

Par exemple, si :

$$
R_{i,t} = 0.03
$$

alors l'action a gagné $3\%$ sur le mois. Si :

$$
R_{i,t} = -0.02
$$

alors elle a perdu $2\%$.

C'est cette variable qui sera utilisée plus tard pour mesurer le momentum sur une fenêtre glissante de 12 mois, en excluant le mois le plus récent.

#### 2. Valeur comptable par action

À partir des bilans trimestriels et annuels, on reconstruit une valeur comptable par action, notée $BVPS_{i,t}$ pour *Book Value Per Share* :

$$
BVPS_{i,t} = \frac{\text{Capitaux propres ordinaires}_{i,t}}{\text{Nombre d'actions en circulation}_{i,t}}
$$

Cette grandeur représente une approximation de la valeur comptable revenant à une action de l'entreprise.

Comme les données comptables sont publiées à fréquence trimestrielle et non mensuelle, on utilise la convention suivante : la dernière valeur comptable disponible est prolongée sur les mois suivants jusqu'à la prochaine publication. On obtient ainsi une série mensuelle exploitable dans un backtest.

#### 3. Ratio cours / valeur comptable : `price_to_book_mensuel_europe.csv`

Une fois le prix de marché $P_{i,t}$ et la valeur comptable par action $BVPS_{i,t}$ obtenus, on calcule le ratio cours / valeur comptable, ou `price-to-book` :

$$
PB_{i,t} = \frac{P_{i,t}}{BVPS_{i,t}}
$$

Ce ratio compare la valorisation boursière de l'action à sa valeur comptable.

- Si $PB_{i,t}$ est élevé, le marché paie cher l'entreprise relativement à ses fonds propres comptables.
- Si $PB_{i,t}$ est faible, l'action est relativement peu chère du point de vue comptable.

Dans une logique de facteur value, on préfère souvent considérer son inverse :

$$
\frac{1}{PB_{i,t}} = \frac{BVPS_{i,t}}{P_{i,t}}
$$

Plus cette quantité est élevée, plus l'action peut être vue comme sous-valorisée relativement à sa valeur comptable.

#### 4. Fichier de couverture : `couverture_price_to_book.csv`

Ce fichier est un fichier de contrôle de qualité des données. Pour chaque ticker, il indique notamment :

- le nombre d'observations comptables disponibles pour reconstruire $BVPS$ ;
- le nombre de mois pour lesquels le ratio $PB_{i,t}$ a pu être calculé.

Il permet donc de vérifier si une action dispose d'un historique fondamental suffisamment riche pour être utilisée dans la stratégie.

#### 5. Table longue finale : `jeu_de_donnees_step0.csv`

Le dernier fichier rassemble toutes les informations dans un format long. Chaque ligne correspond à un couple *(date, ticker)* et contient essentiellement :

$$
(t, i, R_{i,t}, PB_{i,t})
$$

où :

- $t$ désigne la date de fin de mois ;
- $i$ désigne l'action ;
- $R_{i,t}$ est le rendement mensuel ;
- $PB_{i,t}$ est le ratio cours / valeur comptable.

Ce format est particulièrement utile pour la suite, car il permet de calculer à chaque date de rééquilibrage les signaux de momentum et de value, puis de classer l'ensemble des actions de l'univers.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display
from yahooquery import Ticker

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATE_DEBUT = "2007-01-01"
DATE_FIN = "2026-03-31"

# Gere proprement les executions depuis la racine du projet ou depuis le dossier du notebook.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "02_Strategie_long_short").exists() and PROJECT_ROOT.name == "02_Strategie_long_short":
    PROJECT_ROOT = PROJECT_ROOT.parent

DOSSIER_DONNEES = PROJECT_ROOT / "02_Strategie_long_short" / "data"
DOSSIER_DONNEES.mkdir(parents=True, exist_ok=True)

# Univers de grandes capitalisations europeennes, tickers Yahoo Finance.
UNIVERS_EUROPE = {
    "ADS.DE": "Adidas",
    "AI.PA": "Air Liquide",
    "AIR.PA": "Airbus",
    "ALV.DE": "Allianz",
    "ASML.AS": "ASML",
    "BAS.DE": "BASF",
    "BAYN.DE": "Bayer",
    "BN.PA": "Danone",
    "BNP.PA": "BNP Paribas",
    "CS.PA": "AXA",
    "DTE.DE": "Deutsche Telekom",
    "ENEL.MI": "Enel",
    "ENGI.PA": "Engie",
    "IBE.MC": "Iberdrola",
    "IFX.DE": "Infineon",
    "INGA.AS": "ING",
    "MC.PA": "LVMH",
    "MUV2.DE": "Munich Re",
    "NESN.SW": "Nestle",
    "OR.PA": "L'Oreal",
    "PHIA.AS": "Philips",
    "RI.PA": "Pernod Ricard",
    "RMS.PA": "Hermes",
    "SAF.PA": "Safran",
    "SAN.MC": "Banco Santander",
    "SAN.PA": "Sanofi",
    "SAP.DE": "SAP",
    "SHEL.AS": "Shell",
    "SU.PA": "Schneider Electric",
    "TTE.PA": "TotalEnergies",
    "VOW3.DE": "Volkswagen",
}

tickers = list(UNIVERS_EUROPE)
print(f"Nombre de titres dans l'univers : {len(tickers)}")

Nombre de titres dans l'univers : 31


In [14]:
def telecharger_prix_mensuels(tickers, date_debut, date_fin):
    """Telecharge les prix mensuels ajustes, filtre les tickers invalides et calcule les rendements mensuels."""
    prix = yf.download(
        tickers,
        start=date_debut,
        end=date_fin,
        interval="1mo",
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if prix.empty:
        raise ValueError("Aucun prix mensuel n'a pu etre telecharge.")

    if isinstance(prix.columns, pd.MultiIndex):
        close = prix.xs("Close", axis=1, level=1)
    else:
        close = prix[["Close"]].rename(columns={"Close": tickers[0]})

    close.index = pd.to_datetime(close.index).to_period("M").to_timestamp("M")
    close = close.sort_index().dropna(how="all")

    tickers_invalides = [col for col in close.columns if close[col].dropna().empty]
    if tickers_invalides:
        print("Tickers exclus faute de donnees de prix :", ", ".join(tickers_invalides))
        close = close.drop(columns=tickers_invalides)

    if close.empty:
        raise ValueError("Aucun ticker valide ne reste apres filtrage des prix mensuels.")

    rendements = close.pct_change().dropna(how="all")
    return close, rendements, tickers_invalides


def normaliser_bilans_yahoo(bilans):
    """Met en forme la sortie yahooquery pour recuperer un DataFrame avec une colonne symbol."""
    if bilans is None or len(bilans) == 0:
        return pd.DataFrame()

    if isinstance(bilans, dict):
        frames = []
        for ticker, contenu in bilans.items():
            if isinstance(contenu, pd.DataFrame) and not contenu.empty:
                temp = contenu.copy()
                temp["symbol"] = ticker
                frames.append(temp)
        bilans_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    else:
        bilans_df = bilans.reset_index()

    if bilans_df.empty:
        return bilans_df

    if "symbol" not in bilans_df.columns:
        colonnes_index = [col for col in ["ticker", "level_0", "index"] if col in bilans_df.columns]
        if colonnes_index:
            bilans_df = bilans_df.rename(columns={colonnes_index[0]: "symbol"})

    return bilans_df


def extraire_book_value_par_action(balance_sheet):
    """Calcule une valeur comptable par action a partir des bilans Yahoo disponibles."""
    if balance_sheet is None or len(balance_sheet) == 0:
        return pd.Series(dtype=float)

    donnees = balance_sheet.copy()

    colonnes_possibles = [
        "CommonStockEquity",
        "StockholdersEquity",
        "TotalEquityGrossMinorityInterest",
    ]
    colonnes_actions = [
        "OrdinarySharesNumber",
        "ShareIssued",
        "CommonStockSharesOutstanding",
    ]

    colonne_equity = next((col for col in colonnes_possibles if col in donnees.columns), None)
    colonne_actions = next((col for col in colonnes_actions if col in donnees.columns), None)

    if colonne_equity is None or colonne_actions is None:
        return pd.Series(dtype=float)

    donnees = donnees[["asOfDate", colonne_equity, colonne_actions]].copy()
    donnees["asOfDate"] = pd.to_datetime(donnees["asOfDate"])
    donnees = donnees.dropna(subset=[colonne_equity, colonne_actions])
    donnees = donnees[donnees[colonne_actions] != 0]

    if donnees.empty:
        return pd.Series(dtype=float)

    bvps = (
        donnees.sort_values("asOfDate")
        .drop_duplicates(subset="asOfDate", keep="last")
        .set_index("asOfDate")
        .eval(f"{colonne_equity} / {colonne_actions}")
        .astype(float)
    )
    bvps.index = bvps.index.to_period("M").to_timestamp("M")
    return bvps[~bvps.index.duplicated(keep="last")]


def construire_price_to_book_mensuel(tickers, prix_mensuels):
    """Reconstruit un ratio prix / valeur comptable mensuel a partir des bilans trimestriels et annuels."""
    fournisseur = Ticker(tickers, asynchronous=True, validate=True)
    bilans_trimestriels = normaliser_bilans_yahoo(fournisseur.balance_sheet(frequency="q"))
    bilans_annuels = normaliser_bilans_yahoo(fournisseur.balance_sheet(frequency="a"))

    frames = [df for df in [bilans_trimestriels, bilans_annuels] if not df.empty]
    if not frames:
        raise ValueError("Aucune donnee de bilan n'a pu etre recuperee via yahooquery.")

    bilans = pd.concat(frames, ignore_index=True)
    if "symbol" not in bilans.columns:
        raise ValueError("Impossible d'identifier la colonne ticker dans les bilans recuperes.")

    if "asOfDate" in bilans.columns:
        bilans = bilans.sort_values(["symbol", "asOfDate"]).drop_duplicates(subset=["symbol", "asOfDate"], keep="last")

    ratios = {}
    couverture = []

    for ticker in tickers:
        bilan_ticker = bilans.loc[bilans["symbol"] == ticker].copy()
        bvps = extraire_book_value_par_action(bilan_ticker)

        if bvps.empty:
            couverture.append({"ticker": ticker, "book_value_points": 0, "ratio_points": 0})
            continue

        bvps_mensuel = bvps.reindex(prix_mensuels.index, method="ffill")
        ratio = prix_mensuels[ticker] / bvps_mensuel
        ratio = ratio.replace([np.inf, -np.inf], np.nan)
        ratio = ratio.where(ratio > 0)
        ratios[ticker] = ratio

        couverture.append(
            {
                "ticker": ticker,
                "book_value_points": int(bvps.size),
                "ratio_points": int(ratio.notna().sum()),
            }
        )

    ratios_df = pd.DataFrame(ratios, index=prix_mensuels.index).sort_index(axis=1)
    couverture_df = pd.DataFrame(couverture).sort_values("ratio_points", ascending=False)
    return ratios_df, couverture_df


In [15]:
prix_mensuels, rendements_mensuels, tickers_invalides = telecharger_prix_mensuels(tickers, DATE_DEBUT, DATE_FIN)
tickers_valides = prix_mensuels.columns.tolist()
price_to_book_mensuel, couverture_price_to_book = construire_price_to_book_mensuel(tickers_valides, prix_mensuels)

rendements_mensuels = rendements_mensuels.sort_index(axis=1)
price_to_book_mensuel = price_to_book_mensuel.sort_index(axis=1)

rendements_mensuels.to_csv(DOSSIER_DONNEES / "rendements_mensuels_europe.csv", index_label="date")
price_to_book_mensuel.to_csv(DOSSIER_DONNEES / "price_to_book_mensuel_europe.csv", index_label="date")
couverture_price_to_book.to_csv(DOSSIER_DONNEES / "couverture_price_to_book.csv", index=False)

rendements_long = (
    rendements_mensuels.rename_axis(index="date", columns="ticker")
    .stack()
    .rename("monthly_return")
    .reset_index()
)

price_to_book_long = (
    price_to_book_mensuel.rename_axis(index="date", columns="ticker")
    .stack()
    .rename("price_to_book")
    .reset_index()
)

dataset_long = rendements_long.merge(
    price_to_book_long,
    on=["date", "ticker"],
    how="outer",
)
dataset_long["company_name"] = dataset_long["ticker"].map(UNIVERS_EUROPE)
dataset_long = dataset_long[["date", "ticker", "company_name", "monthly_return", "price_to_book"]]
dataset_long.to_csv(DOSSIER_DONNEES / "jeu_de_donnees_step0.csv", index=False)

print("Fichiers generes dans", DOSSIER_DONNEES.resolve())
print(f"Fenetre de travail : {rendements_mensuels.index.min().date()} -> {rendements_mensuels.index.max().date()}")
print(f"Titres valides apres filtrage des prix : {len(tickers_valides)}/{len(tickers)}")
print(f"Titres avec au moins 1 ratio price-to-book disponible : {int((price_to_book_mensuel.notna().sum() > 0).sum())}/{len(tickers_valides)}")
if tickers_invalides:
    print("Tickers exclus :", ", ".join(tickers_invalides))

display(rendements_mensuels.head())
display(price_to_book_mensuel.head())
display(couverture_price_to_book.head(10))
display(dataset_long.head(10))

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SHEL.AS"}}}
$SHEL.AS: possibly delisted; no timezone found

1 Failed download:
['SHEL.AS']: possibly delisted; no timezone found


Tickers exclus faute de donnees de prix : SHEL.AS
Fichiers generes dans C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Fenetre de travail : 2007-02-28 -> 2026-03-31
Titres valides apres filtrage des prix : 30/31
Titres avec au moins 1 ratio price-to-book disponible : 30/30
Tickers exclus : SHEL.AS


Ticker,ADS.DE,AI.PA,AIR.PA,ALV.DE,ASML.AS,BAS.DE,BAYN.DE,BN.PA,BNP.PA,CS.PA,DTE.DE,ENEL.MI,ENGI.PA,IBE.MC,IFX.DE,INGA.AS,MC.PA,MUV2.DE,NESN.SW,OR.PA,PHIA.AS,RI.PA,RMS.PA,SAF.PA,SAN.MC,SAN.PA,SAP.DE,SU.PA,TTE.PA,VOW3.DE
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2007-02-28,0.005417,-0.030386,0.017675,0.063922,-0.039732,0.039497,-0.038274,0.012690,-0.078271,0.000000,0.004445,-0.027709,0.009700,0.025210,0.057429,-0.038152,0.034981,-0.004711,-0.004931,-0.024291,-0.069370,-0.007066,0.052211,-0.047253,-0.039753,-0.047294,-0.016944,-0.009174,-0.015058,0.173416
2007-03-31,0.102640,0.053269,-0.103821,-0.055719,-0.006448,0.096682,0.100529,0.021721,-0.008872,0.000000,-0.087021,0.014566,0.042930,0.066928,0.004310,-0.019213,-0.008121,0.051316,0.042401,0.032857,0.029528,-0.026732,0.048911,0.053633,-0.039356,0.013072,-0.041368,0.050776,0.028028,0.155739
2007-04-30,0.071341,0.001315,0.021533,0.084510,0.081125,0.037375,0.053721,-0.008994,0.095908,0.062500,0.080775,0.041823,-0.004030,0.031365,-0.016309,0.062875,0.035159,0.035384,0.013735,0.078184,0.081176,0.032473,0.024040,-0.025725,-0.012725,0.036559,0.063530,0.095128,0.038322,-0.032326
2007-05-31,0.081870,-0.034421,-0.014755,-0.010198,-0.037018,0.089679,0.082359,-0.039604,0.052509,-0.058823,0.029895,0.013182,0.078612,0.175343,0.005236,0.005584,0.020356,0.103964,0.015739,0.002043,0.043593,0.041723,-0.024889,0.074719,0.082638,0.061944,0.002254,0.030941,0.028278,-0.000972
2007-06-30,-0.004609,0.129672,0.037454,0.076602,0.064935,0.056497,0.048794,0.058913,0.013357,0.033187,0.049925,-0.056771,0.035080,-0.031935,0.068577,-0.009664,-0.012171,-0.026519,-0.023061,0.007860,-0.001266,0.004042,-0.188726,0.004086,-0.026564,-0.161317,0.084579,-0.000645,0.096147,0.054250


,ADS.DE,AI.PA,AIR.PA,ALV.DE,ASML.AS,BAS.DE,BAYN.DE,BN.PA,BNP.PA,CS.PA,DTE.DE,ENEL.MI,ENGI.PA,IBE.MC,IFX.DE,INGA.AS,MC.PA,MUV2.DE,NESN.SW,OR.PA,PHIA.AS,RI.PA,RMS.PA,SAF.PA,SAN.MC,SAN.PA,SAP.DE,SU.PA,TTE.PA,VOW3.DE
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2007-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ticker,book_value_points,ratio_points
13,ENEL.MI,8,52
2,RI.PA,6,46
21,IFX.DE,8,43
1,BNP.PA,7,40
4,VOW3.DE,7,40
3,SAP.DE,7,40
5,TTE.PA,7,40
6,AI.PA,5,40
8,RMS.PA,5,40
7,AIR.PA,7,40


,date,ticker,company_name,monthly_return,price_to_book
0,2007-01-31,ADS.DE,Adidas,NaN,NaN
1,2007-01-31,AI.PA,Air Liquide,NaN,NaN
2,2007-01-31,AIR.PA,Airbus,NaN,NaN
3,2007-01-31,ALV.DE,Allianz,NaN,NaN
4,2007-01-31,ASML.AS,ASML,NaN,NaN
5,2007-01-31,BAS.DE,BASF,NaN,NaN
6,2007-01-31,BAYN.DE,Bayer,NaN,NaN
7,2007-01-31,BN.PA,Danone,NaN,NaN
8,2007-01-31,BNP.PA,BNP Paribas,NaN,NaN
9,2007-01-31,CS.PA,AXA,NaN,NaN


### Résultat de l'étape 0

À l'issue de cette étape, le dossier `02_Strategie_long_short/data` contient :

- `rendements_mensuels_europe.csv` : matrice des rendements mensuels par ticker ;
- `price_to_book_mensuel_europe.csv` : matrice du ratio cours / valeur comptable reconstruit mensuellement ;
- `couverture_price_to_book.csv` : contrôle de couverture des données fondamentales ;
- `jeu_de_donnees_step0.csv` : table longue prête pour le calcul des scores aux étapes suivantes.

L'étape suivante consistera à transformer ces deux composantes en scores momentum et valeur mois par mois.

## Étape 1 - Notation des actions

À cette étape, on transforme les variables brutes de l'étape 0 en signaux comparables d'une action à l'autre.

L'idée est la suivante :

- un titre doit recevoir un score **momentum** s'il a eu de bonnes performances passées sur horizon moyen terme ;
- un titre doit recevoir un score **value** s'il est peu cher relativement à sa valeur comptable ;
- on combine ensuite ces deux dimensions dans un **score global**.

On travaille de manière transversale, c'est-à-dire qu'à chaque date on compare les actions entre elles.

### Formulation mathématique des scores

#### 1. Signal momentum brut

Pour une action $i$ et une date de notation $t$, on calcule d'abord un signal momentum brut comme la moyenne des rendements mensuels observés sur les 12 derniers mois en excluant le mois le plus récent. Dans notre implémentation mensuelle, cela revient à utiliser les rendements de $t-12$ à $t-2$ :

$$
MOM^{raw}_{i,t} = \frac{1}{11} \sum_{k=2}^{12} R_{i,t-k}
$$

L'exclusion du dernier mois permet d'atténuer les effets de retournement de court terme.

#### 2. Standardisation du momentum

Comme les niveaux de momentum brut ne sont pas directement comparables d'une date à l'autre, on standardise le signal de façon transversale, c'est-à-dire en comparant chaque action à toutes les autres actions de l'univers au même mois.

Pour chaque date $t$, le score momentum standardisé est défini par :

$$
Z^{mom}_{i,t} = \frac{MOM^{raw}_{i,t} - \mu^{mom}_t}{\sigma^{mom}_t}
$$

avec :

$$
\mu^{mom}_t = \frac{1}{N_t} \sum_{i=1}^{N_t} MOM^{raw}_{i,t}
$$

et

$$
\sigma^{mom}_t = \sqrt{\frac{1}{N_t} \sum_{i=1}^{N_t} \left(MOM^{raw}_{i,t} - \mu^{mom}_t\right)^2}
$$

où :

- $N_t$ est le nombre d'actions pour lesquelles le signal momentum brut est disponible à la date $t$ ;
- $\mu^{mom}_t$ est la moyenne transversale des signaux momentum bruts à la date $t$ ;
- $\sigma^{mom}_t$ est l'écart-type transversal de ces mêmes signaux à la date $t$.

Autrement dit :

- on calcule d'abord le momentum brut de toutes les actions au mois $t$ ;
- on en prend la moyenne, notée $\mu^{mom}_t$ ;
- on mesure ensuite la dispersion de ces signaux autour de cette moyenne, ce qui donne $\sigma^{mom}_t$ ;
- enfin, on mesure combien l'action $i$ est au-dessus ou au-dessous de la moyenne, en nombre d'écarts-types.

Ainsi :

- si $Z^{mom}_{i,t} > 0$, l'action a un momentum supérieur à la moyenne de l'univers ;
- si $Z^{mom}_{i,t} < 0$, elle a un momentum inférieur à la moyenne ;
- si $Z^{mom}_{i,t} = 1$, son signal momentum est situé à un écart-type au-dessus de la moyenne transversale.

#### 3. Signal value brut

Le facteur value repose sur l'inverse du ratio `price-to-book`, mesuré à la fin du mois précédent :

$$
VAL^{raw}_{i,t} = \frac{1}{PB_{i,t-1}} = \frac{BVPS_{i,t-1}}{P_{i,t-1}}
$$

Plus cette quantité est grande, plus l'action est bon marché relativement à sa valeur comptable.

On le standardise également en coupe transversale :

$$
Z^{val}_{i,t} = \frac{VAL^{raw}_{i,t} - \mu^{val}_t}{\sigma^{val}_t}
$$

avec :

$$
\mu^{val}_t = \frac{1}{N_t^{val}} \sum_{i=1}^{N_t^{val}} VAL^{raw}_{i,t}
$$

et

$$
\sigma^{val}_t = \sqrt{\frac{1}{N_t^{val}} \sum_{i=1}^{N_t^{val}} \left(VAL^{raw}_{i,t} - \mu^{val}_t\right)^2}
$$

où $N_t^{val}$ désigne le nombre d'actions disposant d'un signal value brut à la date $t$.

L'interprétation est la même que pour le momentum :

- un $Z^{val}_{i,t}$ positif indique une action plus attractive que la moyenne sur le critère value ;
- un $Z^{val}_{i,t}$ négatif indique une action moins attractive ;
- plus la valeur absolue du score est grande, plus l'écart à la moyenne de l'univers est marqué.

#### 4. Score global

Enfin, le score global est défini comme la moyenne arithmétique des deux scores standardisés :

$$
Score_{i,t} = \frac{Z^{mom}_{i,t} + Z^{val}_{i,t}}{2}
$$

Une action avec un score global élevé combine donc :

- un bon comportement passé à moyen terme ;
- une valorisation relativement attractive.

À l'inverse, un score très faible indique une action à la fois faible en momentum et peu attractive sur le plan value.

In [16]:
def zscore_transversal(dataframe):
    """Calcule un z-score ligne par ligne pour comparer les titres a une date donnee."""
    moyenne = dataframe.mean(axis=1)
    ecart_type = dataframe.std(axis=1, ddof=0).replace(0, np.nan)
    return dataframe.sub(moyenne, axis=0).div(ecart_type, axis=0)


def calculer_scores_etape_1(rendements_mensuels, price_to_book_mensuel):
    """Construit les scores momentum, value et global a partir des donnees de l'etape 0."""
    momentum_brut = rendements_mensuels.shift(2).rolling(window=11, min_periods=11).mean()
    value_brut = (1 / price_to_book_mensuel).shift(1)

    score_momentum = zscore_transversal(momentum_brut)
    score_value = zscore_transversal(value_brut)
    score_global = (score_momentum + score_value) / 2

    return momentum_brut, value_brut, score_momentum, score_value, score_global


In [17]:
momentum_brut, value_brut, score_momentum, score_value, score_global = calculer_scores_etape_1(
    rendements_mensuels,
    price_to_book_mensuel,
)

score_momentum = score_momentum.sort_index(axis=1)
score_value = score_value.sort_index(axis=1)
score_global = score_global.sort_index(axis=1)

momentum_brut.to_csv(DOSSIER_DONNEES / "momentum_brut_mensuel_europe.csv", index_label="date")
value_brut.to_csv(DOSSIER_DONNEES / "value_brut_mensuel_europe.csv", index_label="date")
score_momentum.to_csv(DOSSIER_DONNEES / "score_momentum_mensuel_europe.csv", index_label="date")
score_value.to_csv(DOSSIER_DONNEES / "score_value_mensuel_europe.csv", index_label="date")
score_global.to_csv(DOSSIER_DONNEES / "score_global_mensuel_europe.csv", index_label="date")

scores_long = (
    score_momentum.rename_axis(index="date", columns="ticker")
    .stack()
    .rename("score_momentum")
    .reset_index()
)
scores_long = scores_long.merge(
    score_value.rename_axis(index="date", columns="ticker").stack().rename("score_value").reset_index(),
    on=["date", "ticker"],
    how="outer",
)
scores_long = scores_long.merge(
    score_global.rename_axis(index="date", columns="ticker").stack().rename("score_global").reset_index(),
    on=["date", "ticker"],
    how="outer",
)
scores_long["company_name"] = scores_long["ticker"].map(UNIVERS_EUROPE)
scores_long = scores_long[["date", "ticker", "company_name", "score_momentum", "score_value", "score_global"]]
scores_long.to_csv(DOSSIER_DONNEES / "scores_etape1.csv", index=False)

premiere_date_valide = score_global.dropna(how="all").index.min()
print("Fichiers generes dans", DOSSIER_DONNEES.resolve())
print(f"Premiere date avec score global exploitable : {premiere_date_valide.date()}")
print(f"Nombre moyen de titres notes par mois : {score_global.notna().sum(axis=1).mean():.1f}")

display(score_momentum.loc[premiere_date_valide].sort_values(ascending=False).head(10).to_frame("score_momentum"))
display(score_value.loc[premiere_date_valide].sort_values(ascending=False).head(10).to_frame("score_value"))
display(score_global.loc[premiere_date_valide].sort_values(ascending=False).head(10).to_frame("score_global"))
display(scores_long.head(10))

Fichiers generes dans C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Premiere date avec score global exploitable : 2022-07-31
Nombre moyen de titres notes par mois : 5.1


,score_momentum
Ticker,
TTE.PA,2.482840
BAYN.DE,1.597205
ENGI.PA,1.069687
IBE.MC,0.856053
SAN.PA,0.758683
DTE.DE,0.719616
CS.PA,0.709173
AI.PA,0.636660
INGA.AS,0.517672


,score_value
ENEL.MI,1.0
RI.PA,-1.0
ADS.DE,NaN
AI.PA,NaN
AIR.PA,NaN
ALV.DE,NaN
ASML.AS,NaN
BAS.DE,NaN
BAYN.DE,NaN
BN.PA,NaN


,score_global
Ticker,
ENEL.MI,0.091065
RI.PA,-0.435396
ADS.DE,NaN
AI.PA,NaN
AIR.PA,NaN
ALV.DE,NaN
ASML.AS,NaN
BAS.DE,NaN
BAYN.DE,NaN


,date,ticker,company_name,score_momentum,score_value,score_global
0,2007-01-31,ADS.DE,Adidas,NaN,NaN,NaN
1,2007-01-31,AI.PA,Air Liquide,NaN,NaN,NaN
2,2007-01-31,AIR.PA,Airbus,NaN,NaN,NaN
3,2007-01-31,ALV.DE,Allianz,NaN,NaN,NaN
4,2007-01-31,ASML.AS,ASML,NaN,NaN,NaN
5,2007-01-31,BAS.DE,BASF,NaN,NaN,NaN
6,2007-01-31,BAYN.DE,Bayer,NaN,NaN,NaN
7,2007-01-31,BN.PA,Danone,NaN,NaN,NaN
8,2007-01-31,BNP.PA,BNP Paribas,NaN,NaN,NaN
9,2007-01-31,CS.PA,AXA,NaN,NaN,NaN


### Résultat de l'étape 1

À l'issue de cette étape, le dossier `02_Strategie_long_short/data` contient également :

- `momentum_brut_mensuel_europe.csv` : la moyenne glissante des rendements passés, hors dernier mois ;
- `value_brut_mensuel_europe.csv` : l'inverse du ratio `price-to-book` décalé d'un mois ;
- `score_momentum_mensuel_europe.csv` : le z-score transversal du signal momentum ;
- `score_value_mensuel_europe.csv` : le z-score transversal du signal value ;
- `score_global_mensuel_europe.csv` : la moyenne des deux scores standardisés ;
- `scores_etape1.csv` : une table longue récapitulant l'ensemble des scores par date et par action.

L'étape suivante consistera à sélectionner, à chaque date, les 15 meilleures actions pour le portefeuille long et les 15 moins bonnes pour le portefeuille short.

## Étape 2 - Sélection des actions

Une fois les scores calculés, on passe à la sélection des titres à chaque date de rééquilibrage.

Le principe est simple :

- le portefeuille **long** retient les actions ayant les meilleurs scores globaux ;
- le portefeuille **short** retient les actions ayant les scores globaux les plus faibles.

Cette étape transforme donc un signal de classement en une liste concrète de titres achetés et vendus à découvert.

### Formulation mathématique de la sélection

Pour chaque date de rééquilibrage $t$, on dispose d'un score global $Score_{i,t}$ pour chaque action $i$ de l'univers.

On classe alors les actions par ordre décroissant de score.

#### 1. Portefeuille long

On définit l'ensemble des titres longs comme les 15 actions ayant les scores globaux les plus élevés :

$$
\mathcal{L}_t = \text{Top}_{15}\left(Score_{i,t}\right)
$$

Autrement dit, si l'on ordonne les actions de la meilleure à la moins bonne selon $Score_{i,t}$, alors $\mathcal{L}_t$ contient les 15 premières.

#### 2. Portefeuille short

On définit l'ensemble des titres short comme les 15 actions ayant les scores globaux les plus faibles :

$$
\mathcal{S}_t = \text{Bottom}_{15}\left(Score_{i,t}\right)
$$

Ce sont donc les 15 dernières actions du classement à la date $t$.

#### 3. Lecture économique

À chaque date :

- le portefeuille long est composé des titres combinant le mieux les dimensions momentum et value ;
- le portefeuille short est composé des titres combinant le moins bien ces deux dimensions.

On cherche ainsi à capter l'écart de performance entre les meilleures et les moins bonnes actions selon le signal composite.

#### 4. Remarque pratique

La sélection est réalisée uniquement parmi les actions dont le score global est disponible. Si, à une date donnée, certaines actions n'ont pas de score exploitable, elles ne participent pas au classement de ce mois.

In [18]:
def selectionner_portefeuilles(score_global, univers, nb_positions=15):
    """Construit des selections long et short disjointes a partir du score global mensuel."""
    selections_long = []
    selections_short = []

    for date, scores in score_global.iterrows():
        scores_valides = scores.dropna().sort_values(ascending=False)
        if len(scores_valides) < 2 * nb_positions:
            continue

        long_t = scores_valides.head(nb_positions)
        short_t = scores_valides.iloc[-nb_positions:].sort_values(ascending=True)

        for rang, (ticker, score) in enumerate(long_t.items(), start=1):
            selections_long.append(
                {
                    "date": date,
                    "ticker": ticker,
                    "company_name": univers.get(ticker),
                    "score_global": score,
                    "rang": rang,
                    "side": "long",
                }
            )

        for rang, (ticker, score) in enumerate(short_t.items(), start=1):
            selections_short.append(
                {
                    "date": date,
                    "ticker": ticker,
                    "company_name": univers.get(ticker),
                    "score_global": score,
                    "rang": rang,
                    "side": "short",
                }
            )

    long_df = pd.DataFrame(selections_long)
    short_df = pd.DataFrame(selections_short)
    return long_df, short_df


In [19]:
portefeuille_long, portefeuille_short = selectionner_portefeuilles(
    score_global,
    UNIVERS_EUROPE,
    nb_positions=15,
)

selection_etape2 = pd.concat([portefeuille_long, portefeuille_short], ignore_index=True)
selection_etape2 = selection_etape2.sort_values(["date", "side", "rang"]).reset_index(drop=True)

portefeuille_long.to_csv(DOSSIER_DONNEES / "selection_long_etape2.csv", index=False)
portefeuille_short.to_csv(DOSSIER_DONNEES / "selection_short_etape2.csv", index=False)
selection_etape2.to_csv(DOSSIER_DONNEES / "selection_etape2.csv", index=False)

resume_selection = selection_etape2.groupby(["date", "side"]).size().unstack(fill_value=0)
resume_selection.to_csv(DOSSIER_DONNEES / "resume_selection_etape2.csv", index_label="date")

premiere_date_selection = selection_etape2["date"].min()
print("Fichiers generes dans", DOSSIER_DONNEES.resolve())
print(f"Premiere date de selection : {premiere_date_selection.date()}")
print(f"Nombre moyen de titres long par mois : {resume_selection['long'].mean():.1f}")
print(f"Nombre moyen de titres short par mois : {resume_selection['short'].mean():.1f}")

display(portefeuille_long[portefeuille_long["date"] == premiere_date_selection].head(15))
display(portefeuille_short[portefeuille_short["date"] == premiere_date_selection].head(15))
display(resume_selection.head())
display(selection_etape2.head(20))

Fichiers generes dans C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Premiere date de selection : 2023-01-31
Nombre moyen de titres long par mois : 15.0
Nombre moyen de titres short par mois : 15.0


,date,ticker,company_name,score_global,rang,side
0,2023-01-31,VOW3.DE,Volkswagen,1.695210,1,long
1,2023-01-31,TTE.PA,TotalEnergies,1.004950,2,long
2,2023-01-31,SAN.MC,Banco Santander,0.926817,3,long
3,2023-01-31,ENGI.PA,Engie,0.914747,4,long
4,2023-01-31,BNP.PA,BNP Paribas,0.909819,5,long
5,2023-01-31,INGA.AS,ING,0.704319,6,long
6,2023-01-31,BAYN.DE,Bayer,0.618802,7,long
7,2023-01-31,MUV2.DE,Munich Re,0.463467,8,long
8,2023-01-31,DTE.DE,Deutsche Telekom,0.411115,9,long
9,2023-01-31,CS.PA,AXA,0.390064,10,long


,date,ticker,company_name,score_global,rang,side
0,2023-01-31,ADS.DE,Adidas,-1.682903,1,short
1,2023-01-31,PHIA.AS,Philips,-1.375448,2,short
2,2023-01-31,ASML.AS,ASML,-0.723959,3,short
3,2023-01-31,OR.PA,L'Oreal,-0.623804,4,short
4,2023-01-31,NESN.SW,Nestle,-0.560984,5,short
5,2023-01-31,SU.PA,Schneider Electric,-0.522729,6,short
6,2023-01-31,IFX.DE,Infineon,-0.515325,7,short
7,2023-01-31,ENEL.MI,Enel,-0.503280,8,short
8,2023-01-31,SAP.DE,SAP,-0.493342,9,short
9,2023-01-31,RI.PA,Pernod Ricard,-0.406840,10,short


side,long,short
date,,
2023-01-31,15,15
2023-02-28,15,15
2023-03-31,15,15
2023-04-30,15,15
2023-05-31,15,15


,date,ticker,company_name,score_global,rang,side
0,2023-01-31,VOW3.DE,Volkswagen,1.695210,1,long
1,2023-01-31,TTE.PA,TotalEnergies,1.004950,2,long
2,2023-01-31,SAN.MC,Banco Santander,0.926817,3,long
3,2023-01-31,ENGI.PA,Engie,0.914747,4,long
4,2023-01-31,BNP.PA,BNP Paribas,0.909819,5,long
5,2023-01-31,INGA.AS,ING,0.704319,6,long
6,2023-01-31,BAYN.DE,Bayer,0.618802,7,long
7,2023-01-31,MUV2.DE,Munich Re,0.463467,8,long
8,2023-01-31,DTE.DE,Deutsche Telekom,0.411115,9,long
9,2023-01-31,CS.PA,AXA,0.390064,10,long


### Résultat de l'étape 2

À l'issue de cette étape, le dossier `02_Strategie_long_short/data` contient également :

- `selection_long_etape2.csv` : les 15 actions retenues côté long à chaque date ;
- `selection_short_etape2.csv` : les 15 actions retenues côté short à chaque date ;
- `selection_etape2.csv` : la table complète des sélections long et short ;
- `resume_selection_etape2.csv` : un résumé du nombre de titres sélectionnés par date et par côté.

L'étape suivante consistera à pondérer ces titres à l'intérieur des portefeuilles long et short en fonction de la valeur absolue de leurs scores globaux.

## Étape 3 - Construction du portefeuille

Après avoir sélectionné les actions à acheter et à vendre, il faut maintenant déterminer leur poids à l'intérieur des portefeuilles long et short.

La règle imposée est la suivante :

- à l'intérieur de chaque côté du portefeuille, les poids sont proportionnels à la valeur absolue du score global ;
- le portefeuille long représente 100% d'exposition longue ;
- le portefeuille short représente 100% d'exposition short.

L'intuition est qu'un titre dont le score est très extrême doit recevoir un poids plus important qu'un titre sélectionné de justesse.

### Formulation mathématique des pondérations

Pour chaque date de rééquilibrage $t$, on dispose de deux ensembles de titres :

- l'ensemble long $\mathcal{L}_t$ ;
- l'ensemble short $\mathcal{S}_t$.

#### 1. Pondérations du portefeuille long

Pour une action $i \in \mathcal{L}_t$, le poids long est défini par :

$$
w^{L}_{i,t} = \frac{|Score_{i,t}|}{\sum_{j \in \mathcal{L}_t} |Score_{j,t}|}
$$

Ainsi, plus le score global de l'action est élevé en valeur absolue, plus son poids dans le portefeuille long est important.

Par construction, la somme des poids longs vaut :

$$
\sum_{i \in \mathcal{L}_t} w^{L}_{i,t} = 1
$$

#### 2. Pondérations du portefeuille short

Pour une action $i \in \mathcal{S}_t$, le poids short en valeur absolue est défini par :

$$
w^{S}_{i,t} = \frac{|Score_{i,t}|}{\sum_{j \in \mathcal{S}_t} |Score_{j,t}|}
$$

Là encore, la somme des poids vaut 1 en valeur absolue :

$$
\sum_{i \in \mathcal{S}_t} w^{S}_{i,t} = 1
$$

Si l'on souhaite représenter l'exposition réelle du portefeuille short, on peut associer un poids signé négatif :

$$
\tilde{w}^{S}_{i,t} = - w^{S}_{i,t}
$$

#### 3. Performance mensuelle de la stratégie

Les pondérations sont construites à la fin du mois $t$, puis appliquées aux rendements réalisés sur le mois suivant $t+1$. Cela permet d'éviter tout biais d'anticipation.

La performance de la jambe longue entre $t$ et $t+1$ est donc :

$$
r^{L}_{t+1} = \sum_{i \in \mathcal{L}_t} w^{L}_{i,t} \; R_{i,t+1}
$$

La performance de la jambe short s'écrit, avec des poids signés négatifs :

$$
r^{S}_{t+1} = \sum_{i \in \mathcal{S}_t} \tilde{w}^{S}_{i,t} \; R_{i,t+1}
$$

Comme $\tilde{w}^{S}_{i,t} = -w^{S}_{i,t}$, cette performance devient positive lorsque les titres shortés baissent effectivement sur le mois suivant.

La performance totale de la stratégie long-short est alors :

$$
r^{LS}_{t+1} = r^{L}_{t+1} + r^{S}_{t+1}
$$

#### 4. Lecture économique

Cette construction donne davantage de poids :

- aux titres les plus fortement favorisés par le signal dans le portefeuille long ;
- aux titres les plus fortement défavorisés par le signal dans le portefeuille short.

Le portefeuille final est donc long 100% et short 100%, ce qui correspond à une stratégie long-short avec une jambe acheteuse et une jambe vendeuse de même intensité nominale.

In [20]:
def calculer_ponderations_portefeuille(selection_df):
    """Calcule les poids proportionnels a la valeur absolue du score global."""
    selection_df = selection_df.copy()
    selection_df["score_absolu"] = selection_df["score_global"].abs()

    somme_scores = selection_df.groupby(["date", "side"])["score_absolu"].transform("sum")
    selection_df["poids_absolu"] = selection_df["score_absolu"] / somme_scores
    selection_df["poids_signe"] = np.where(
        selection_df["side"] == "long",
        selection_df["poids_absolu"],
        -selection_df["poids_absolu"],
    )
    return selection_df


def calculer_performance_mensuelle_strategie(ponderations_df, rendements_mensuels):
    """Applique les poids formes en t aux rendements observes en t+1."""
    rendements_long = (
        rendements_mensuels.rename_axis(index="date_reelle", columns="ticker")
        .stack()
        .rename("rendement_realise")
        .reset_index()
    )

    calendrier = pd.DataFrame(
        {
            "date_signal": rendements_mensuels.index[:-1],
            "date_reelle": rendements_mensuels.index[1:],
        }
    )

    performances = ponderations_df.merge(calendrier, left_on="date", right_on="date_signal", how="inner")
    performances = performances.merge(rendements_long, on=["date_reelle", "ticker"], how="left")
    performances["contribution"] = performances["poids_signe"] * performances["rendement_realise"]

    resume = (
        performances.groupby(["date_signal", "date_reelle", "side"], as_index=False)["contribution"]
        .sum()
        .pivot(index=["date_signal", "date_reelle"], columns="side", values="contribution")
        .reset_index()
        .rename_axis(columns=None)
    )

    if "long" not in resume.columns:
        resume["long"] = np.nan
    if "short" not in resume.columns:
        resume["short"] = np.nan

    resume = resume.rename(
        columns={
            "date_signal": "date_formation",
            "date_reelle": "date_performance",
            "long": "performance_long",
            "short": "performance_short",
        }
    )
    resume["performance_strategie"] = resume["performance_long"] + resume["performance_short"]
    return performances, resume.sort_values("date_performance").reset_index(drop=True)


In [21]:
ponderations_etape3 = calculer_ponderations_portefeuille(selection_etape2)

ponderations_long = ponderations_etape3[ponderations_etape3["side"] == "long"].copy()
ponderations_short = ponderations_etape3[ponderations_etape3["side"] == "short"].copy()

ponderations_etape3.to_csv(DOSSIER_DONNEES / "ponderations_etape3.csv", index=False)
ponderations_long.to_csv(DOSSIER_DONNEES / "ponderations_long_etape3.csv", index=False)
ponderations_short.to_csv(DOSSIER_DONNEES / "ponderations_short_etape3.csv", index=False)

resume_ponderations = ponderations_etape3.groupby(["date", "side"])["poids_absolu"].sum().unstack(fill_value=0)
resume_ponderations.to_csv(DOSSIER_DONNEES / "resume_ponderations_etape3.csv", index_label="date")

performances_detail, performances_mensuelles = calculer_performance_mensuelle_strategie(
    ponderations_etape3,
    rendements_mensuels,
)
performances_detail.to_csv(DOSSIER_DONNEES / "performances_detail_etape3.csv", index=False)
performances_mensuelles.to_csv(DOSSIER_DONNEES / "performances_mensuelles_strategie.csv", index=False)

premiere_date_ponderation = ponderations_etape3["date"].min()
premiere_date_performance = performances_mensuelles["date_performance"].min()
print("Fichiers generes dans", DOSSIER_DONNEES.resolve())
print(f"Premiere date de ponderation : {premiere_date_ponderation.date()}")
print(f"Premiere date de performance realisee : {premiere_date_performance.date()}")
print(f"Somme moyenne des poids long : {resume_ponderations['long'].mean():.4f}")
print(f"Somme moyenne des poids short : {resume_ponderations['short'].mean():.4f}")
print(f"Performance moyenne mensuelle de la strategie : {performances_mensuelles['performance_strategie'].mean():.4%}")

display(ponderations_long[ponderations_long["date"] == premiere_date_ponderation].head(15))
display(ponderations_short[ponderations_short["date"] == premiere_date_ponderation].head(15))
display(resume_ponderations.head())
display(performances_mensuelles.head())
display(performances_detail.head(20))

Fichiers generes dans C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Premiere date de ponderation : 2023-01-31
Premiere date de performance realisee : 2023-02-28
Somme moyenne des poids long : 1.0000
Somme moyenne des poids short : 1.0000
Performance moyenne mensuelle de la strategie : 1.5080%


,date,ticker,company_name,score_global,rang,side,score_absolu,poids_absolu,poids_signe
0,2023-01-31,VOW3.DE,Volkswagen,1.695210,1,long,1.695210,0.198138,0.198138
1,2023-01-31,TTE.PA,TotalEnergies,1.004950,2,long,1.004950,0.117459,0.117459
2,2023-01-31,SAN.MC,Banco Santander,0.926817,3,long,0.926817,0.108327,0.108327
3,2023-01-31,ENGI.PA,Engie,0.914747,4,long,0.914747,0.106916,0.106916
4,2023-01-31,BNP.PA,BNP Paribas,0.909819,5,long,0.909819,0.106340,0.106340
5,2023-01-31,INGA.AS,ING,0.704319,6,long,0.704319,0.082321,0.082321
6,2023-01-31,BAYN.DE,Bayer,0.618802,7,long,0.618802,0.072326,0.072326
7,2023-01-31,MUV2.DE,Munich Re,0.463467,8,long,0.463467,0.054170,0.054170
8,2023-01-31,DTE.DE,Deutsche Telekom,0.411115,9,long,0.411115,0.048051,0.048051
9,2023-01-31,CS.PA,AXA,0.390064,10,long,0.390064,0.045591,0.045591


,date,ticker,company_name,score_global,rang,side,score_absolu,poids_absolu,poids_signe
15,2023-01-31,ADS.DE,Adidas,-1.682903,1,short,1.682903,0.199441,-0.199441
16,2023-01-31,PHIA.AS,Philips,-1.375448,2,short,1.375448,0.163004,-0.163004
17,2023-01-31,ASML.AS,ASML,-0.723959,3,short,0.723959,0.085796,-0.085796
18,2023-01-31,OR.PA,L'Oreal,-0.623804,4,short,0.623804,0.073927,-0.073927
19,2023-01-31,NESN.SW,Nestle,-0.560984,5,short,0.560984,0.066482,-0.066482
20,2023-01-31,SU.PA,Schneider Electric,-0.522729,6,short,0.522729,0.061949,-0.061949
21,2023-01-31,IFX.DE,Infineon,-0.515325,7,short,0.515325,0.061071,-0.061071
22,2023-01-31,ENEL.MI,Enel,-0.503280,8,short,0.503280,0.059644,-0.059644
23,2023-01-31,SAP.DE,SAP,-0.493342,9,short,0.493342,0.058466,-0.058466
24,2023-01-31,RI.PA,Pernod Ricard,-0.406840,10,short,0.406840,0.048215,-0.048215


side,long,short
date,,
2023-01-31,1.0,1.0
2023-02-28,1.0,1.0
2023-03-31,1.0,1.0
2023-04-30,1.0,1.0
2023-05-31,1.0,1.0


,date_formation,date_performance,performance_long,performance_short,performance_strategie
0,2023-01-31,2023-02-28,0.041367,0.008645,0.050012
1,2023-02-28,2023-03-31,-0.041083,-0.087645,-0.128728
2,2023-03-31,2023-04-30,0.020952,-0.038751,-0.017799
3,2023-04-30,2023-05-31,-0.026505,0.023943,-0.002563
4,2023-05-31,2023-06-30,0.092012,-0.069848,0.022164


,date,ticker,company_name,score_global,rang,side,score_absolu,poids_absolu,poids_signe,date_signal,date_reelle,rendement_realise,contribution
0,2023-01-31,VOW3.DE,Volkswagen,1.695210,1,long,1.695210,0.198138,0.198138,2023-01-31,2023-02-28,0.017489,0.003465
1,2023-01-31,TTE.PA,TotalEnergies,1.004950,2,long,1.004950,0.117459,0.117459,2023-01-31,2023-02-28,0.040141,0.004715
2,2023-01-31,SAN.MC,Banco Santander,0.926817,3,long,0.926817,0.108327,0.108327,2023-01-31,2023-02-28,0.162192,0.017570
3,2023-01-31,ENGI.PA,Engie,0.914747,4,long,0.914747,0.106916,0.106916,2023-01-31,2023-02-28,0.062673,0.006701
4,2023-01-31,BNP.PA,BNP Paribas,0.909819,5,long,0.909819,0.106340,0.106340,2023-01-31,2023-02-28,0.051669,0.005495
5,2023-01-31,INGA.AS,ING,0.704319,6,long,0.704319,0.082321,0.082321,2023-01-31,2023-02-28,0.007764,0.000639
6,2023-01-31,BAYN.DE,Bayer,0.618802,7,long,0.618802,0.072326,0.072326,2023-01-31,2023-02-28,-0.012456,-0.000901
7,2023-01-31,MUV2.DE,Munich Re,0.463467,8,long,0.463467,0.054170,0.054170,2023-01-31,2023-02-28,-0.014804,-0.000802
8,2023-01-31,DTE.DE,Deutsche Telekom,0.411115,9,long,0.411115,0.048051,0.048051,2023-01-31,2023-02-28,0.038386,0.001845
9,2023-01-31,CS.PA,AXA,0.390064,10,long,0.390064,0.045591,0.045591,2023-01-31,2023-02-28,0.043159,0.001968


### Résultat de l'étape 3

À l'issue de cette étape, le dossier `02_Strategie_long_short/data` contient également :

- `ponderations_etape3.csv` : la table complète des positions et de leurs poids ;
- `ponderations_long_etape3.csv` : les pondérations des titres du portefeuille long ;
- `ponderations_short_etape3.csv` : les pondérations des titres du portefeuille short ;
- `resume_ponderations_etape3.csv` : la somme des poids par date et par côté pour contrôle ;
- `performances_detail_etape3.csv` : le détail des contributions par titre à la performance mensuelle ;
- `performances_mensuelles_strategie.csv` : la performance mensuelle de la jambe longue, de la jambe short et de la stratégie globale.

Le notebook contient désormais les briques complètes de la stratégie : données, scores, sélection, pondérations et performances mensuelles. On peut maintenant évaluer la pertinence de cette stratégie face à des portefeuilles long/short tirés au hasard.

## Partie 2 - Évaluation de la stratégie

L'objectif est de mesurer si la stratégie construite précédemment apporte réellement de l'information, ou si des performances comparables pourraient être obtenues simplement par chance.

Pour cela, on compare la stratégie factorielle à un grand nombre de stratégies long/short aléatoires, appelées ici **stratégies lucky**.

À chaque date de rééquilibrage, une stratégie lucky :

- sélectionne aléatoirement 15 actions pour la jambe longue ;
- sélectionne aléatoirement 15 autres actions, disjointes, pour la jambe short ;
- conserve la même structure long 100% / short 100% que la stratégie étudiée.

On répète cette expérience un grand nombre de fois, puis on compare les statistiques obtenues.

### Méthodologie et formules de comparaison

#### 1. Univers disponible à chaque date

À la date de formation $t$, on note $\mathcal{U}_t$ l'ensemble des actions pour lesquelles on dispose :

- d'un score exploitable à la date $t$ ;
- d'un rendement observé au mois suivant $t+1$.

Les stratégies lucky sont construites sur ce même univers afin d'assurer une comparaison équitable.

#### 2. Construction d'une stratégie lucky

Pour une simulation lucky $b$, on tire deux sous-ensembles disjoints de taille 15 :

$$
\mathcal{L}^{(b)}_t \subset \mathcal{U}_t, \qquad \mathcal{S}^{(b)}_t \subset \mathcal{U}_t, \qquad \mathcal{L}^{(b)}_t \cap \mathcal{S}^{(b)}_t = \varnothing
$$

avec :

$$
|\mathcal{L}^{(b)}_t| = |\mathcal{S}^{(b)}_t| = 15
$$

Dans cette évaluation, chaque jambe lucky est équipondérée.

#### 3. Performance d'une stratégie lucky

La performance mensuelle d'une simulation $b$ entre $t$ et $t+1$ s'écrit alors :

$$
r^{(b)}_{t+1} = \frac{1}{15} \sum_{i \in \mathcal{L}^{(b)}_t} R_{i,t+1} - \frac{1}{15} \sum_{i \in \mathcal{S}^{(b)}_t} R_{i,t+1}
$$

#### 4. Indicateurs étudiés

On compare ensuite la stratégie factorielle et les stratégies lucky à l'aide des indicateurs suivants :

- la performance moyenne mensuelle ;
- la volatilité mensuelle ;
- la performance annualisée ;
- la volatilité annualisée ;
- le ratio de Sharpe annualisé ;
- la performance cumulée ;
- le taux de mois positifs ;
- le maximum drawdown.

Enfin, on calcule le percentile de la stratégie réelle dans la distribution lucky. Un percentile élevé signifie que la stratégie réelle domine une grande proportion des stratégies aléatoires.

In [22]:
def calculer_statistiques_performance(serie_rendements):
    """Calcule des statistiques de performance a partir d'une serie mensuelle."""
    serie = pd.Series(serie_rendements).dropna()
    if serie.empty:
        return {
            "nb_mois": 0,
            "performance_moyenne_mensuelle": np.nan,
            "volatilite_mensuelle": np.nan,
            "performance_annualisee": np.nan,
            "volatilite_annualisee": np.nan,
            "sharpe_annualise": np.nan,
            "performance_cumulee": np.nan,
            "taux_mois_positifs": np.nan,
            "max_drawdown": np.nan,
        }

    perf_moy = serie.mean()
    vol_mens = serie.std(ddof=0)
    perf_ann = (1 + perf_moy) ** 12 - 1
    vol_ann = vol_mens * np.sqrt(12)
    sharpe = np.nan if vol_ann == 0 else perf_ann / vol_ann
    wealth = (1 + serie).cumprod()
    drawdown = wealth / wealth.cummax() - 1

    return {
        "nb_mois": int(serie.size),
        "performance_moyenne_mensuelle": perf_moy,
        "volatilite_mensuelle": vol_mens,
        "performance_annualisee": perf_ann,
        "volatilite_annualisee": vol_ann,
        "sharpe_annualise": sharpe,
        "performance_cumulee": wealth.iloc[-1] - 1,
        "taux_mois_positifs": (serie > 0).mean(),
        "max_drawdown": drawdown.min(),
    }


def simuler_strategies_lucky(score_global, rendements_mensuels, nb_positions=15, nb_simulations=1000, seed=42):
    """Simule des strategies long/short aleatoires sur le meme univers disponible a chaque date."""
    rng = np.random.default_rng(seed)
    dates_utiles = []
    univers_par_date = {}

    for idx in range(len(rendements_mensuels.index) - 1):
        date_signal = rendements_mensuels.index[idx]
        date_reelle = rendements_mensuels.index[idx + 1]
        univers_score = set(score_global.loc[date_signal].dropna().index)
        univers_rendement = set(rendements_mensuels.loc[date_reelle].dropna().index)
        univers = sorted(univers_score & univers_rendement)
        if len(univers) >= 2 * nb_positions:
            dates_utiles.append((date_signal, date_reelle))
            univers_par_date[date_signal] = univers

    details = []
    resumes = []

    for simulation in range(nb_simulations):
        rendements_simulation = []
        for date_signal, date_reelle in dates_utiles:
            univers = univers_par_date[date_signal]
            tirage = rng.choice(univers, size=2 * nb_positions, replace=False)
            long_names = tirage[:nb_positions]
            short_names = tirage[nb_positions:]

            perf_long = float(rendements_mensuels.loc[date_reelle, long_names].mean())
            perf_short = float(-rendements_mensuels.loc[date_reelle, short_names].mean())
            perf_total = perf_long + perf_short
            rendements_simulation.append(perf_total)

            details.append(
                {
                    "simulation": simulation,
                    "date_formation": date_signal,
                    "date_performance": date_reelle,
                    "performance_long": perf_long,
                    "performance_short": perf_short,
                    "performance_strategie": perf_total,
                }
            )

        stats = calculer_statistiques_performance(pd.Series(rendements_simulation))
        stats["simulation"] = simulation
        resumes.append(stats)

    return pd.DataFrame(details), pd.DataFrame(resumes)


def comparer_strategie_a_lucky(performances_mensuelles, lucky_stats):
    """Compare la strategie factorielle a la distribution des simulations lucky."""
    stats_strategie = calculer_statistiques_performance(performances_mensuelles["performance_strategie"])
    lignes = []
    metriques = [
        "performance_moyenne_mensuelle",
        "volatilite_mensuelle",
        "performance_annualisee",
        "volatilite_annualisee",
        "sharpe_annualise",
        "performance_cumulee",
        "taux_mois_positifs",
        "max_drawdown",
    ]
    higher_is_better = {
        "performance_moyenne_mensuelle",
        "performance_annualisee",
        "sharpe_annualise",
        "performance_cumulee",
        "taux_mois_positifs",
    }

    for metrique in metriques:
        distribution = lucky_stats[metrique].dropna()
        valeur = stats_strategie[metrique]
        percentile = (distribution <= valeur).mean() if metrique in higher_is_better else (distribution >= valeur).mean()
        lignes.append(
            {
                "metrique": metrique,
                "strategie": valeur,
                "lucky_moyenne": distribution.mean(),
                "lucky_mediane": distribution.median(),
                "lucky_pct05": distribution.quantile(0.05),
                "lucky_pct95": distribution.quantile(0.95),
                "percentile_strategie": percentile,
            }
        )

    return pd.DataFrame(lignes), pd.DataFrame([stats_strategie])


In [23]:
NB_SIMULATIONS_LUCKY = 1000
SEED_LUCKY = 42

performances_detail_lucky, lucky_stats = simuler_strategies_lucky(
    score_global,
    rendements_mensuels,
    nb_positions=15,
    nb_simulations=NB_SIMULATIONS_LUCKY,
    seed=SEED_LUCKY,
)

evaluation_resume, stats_strategie = comparer_strategie_a_lucky(
    performances_mensuelles,
    lucky_stats,
)

performances_detail_lucky.to_csv(DOSSIER_DONNEES / "performances_lucky_detail.csv", index=False)
lucky_stats.to_csv(DOSSIER_DONNEES / "performances_lucky_stats.csv", index=False)
evaluation_resume.to_csv(DOSSIER_DONNEES / "evaluation_strategie_vs_lucky.csv", index=False)
stats_strategie.to_csv(DOSSIER_DONNEES / "statistiques_strategie.csv", index=False)

print("Fichiers generes dans", DOSSIER_DONNEES.resolve())
print(f"Nombre de simulations lucky : {NB_SIMULATIONS_LUCKY}")
print(f"Nombre de mois evalues pour la strategie : {len(performances_mensuelles)}")

display(stats_strategie)
display(evaluation_resume)
display(lucky_stats.head())

Fichiers generes dans C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Nombre de simulations lucky : 1000
Nombre de mois evalues pour la strategie : 38


,nb_mois,performance_moyenne_mensuelle,volatilite_mensuelle,performance_annualisee,volatilite_annualisee,sharpe_annualise,performance_cumulee,taux_mois_positifs,max_drawdown
0,38,0.01508,0.044948,0.196751,0.155704,1.263624,0.69992,0.684211,-0.149419


,metrique,strategie,lucky_moyenne,lucky_mediane,lucky_pct05,lucky_pct95,percentile_strategie
0,performance_moyenne_mensuelle,0.015080,0.000085,0.000174,-0.006255,0.006515,1.000
1,volatilite_mensuelle,0.044948,0.022602,0.022623,0.018141,0.027422,0.000
2,performance_annualisee,0.196751,0.001972,0.002092,-0.072532,0.081042,1.000
3,volatilite_annualisee,0.155704,0.078294,0.078368,0.062843,0.094994,0.000
4,sharpe_annualise,1.263624,0.020996,0.026364,-0.949189,1.090653,0.972
5,performance_cumulee,0.699920,0.003468,-0.004776,-0.219317,0.269035,1.000
6,taux_mois_positifs,0.684211,0.500711,0.500000,0.368421,0.631579,0.992
7,max_drawdown,-0.149419,-0.140292,-0.128278,-0.254561,-0.060986,0.619


,nb_mois,performance_moyenne_mensuelle,volatilite_mensuelle,performance_annualisee,volatilite_annualisee,sharpe_annualise,performance_cumulee,taux_mois_positifs,max_drawdown,simulation
0,38,-0.002333,0.023923,-0.027634,0.082871,-0.333461,-0.094959,0.552632,-0.148760,0
1,38,-0.003658,0.023985,-0.043027,0.083085,-0.517861,-0.139559,0.500000,-0.160496,1
2,38,-0.004130,0.023088,-0.048451,0.079978,-0.605799,-0.154295,0.421053,-0.195866,2
3,38,-0.000899,0.028039,-0.010730,0.097129,-0.110469,-0.047817,0.500000,-0.203734,3
4,38,-0.005929,0.019593,-0.068875,0.067872,-1.014788,-0.208092,0.368421,-0.230862,4


### Résultat de la Partie 2

Le dossier `data` contient maintenant aussi :

- `performances_lucky_detail.csv` : les performances mensuelles de chaque simulation aléatoire ;
- `performances_lucky_stats.csv` : les statistiques simulation par simulation ;
- `evaluation_strategie_vs_lucky.csv` : le tableau comparatif entre la stratégie et la distribution lucky ;
- `statistiques_strategie.csv` : les statistiques résumées de la stratégie factorielle.

Cette partie permet de juger si la stratégie semble réellement informative ou si ses résultats ressemblent à ceux qu'on pourrait obtenir simplement par tirage aléatoire.

## Rendu final

Pour simplifier l'exploitation des résultats, on exporte ici un rendu final uniquement en fichiers CSV.

On filtre les données sur la fenêtre demandée, puis on génère plusieurs fichiers plats faciles à ouvrir, trier et réutiliser.

### Contenu des exports CSV

Le rendu final regroupe les objets suivants :

- un fichier de performances mensuelles de la stratégie long-short ;
- un fichier des pondérations détaillées par titre ;
- un fichier des pondérations du portefeuille long ;
- un fichier des pondérations du portefeuille short ;
- un fichier de comparaison avec les stratégies lucky ;
- un fichier de statistiques résumées de la stratégie.

La performance mensuelle exportée correspond à :

$$
r^{LS}_{t} = r^{L}_{t} + r^{S}_{t}
$$

et les pondérations exportées sont celles formées à la date de rééquilibrage précédente et appliquées au mois de performance suivant.

In [24]:
DATE_EXPORT_DEBUT = "2008-03-01"
DATE_EXPORT_FIN = "2023-03-31"
PREFIXE_EXPORT_FINAL = "rendu_final"

debut_export = pd.Timestamp(DATE_EXPORT_DEBUT)
fin_export = pd.Timestamp(DATE_EXPORT_FIN)

performances_export = performances_mensuelles.loc[
    (performances_mensuelles["date_performance"] >= debut_export)
    & (performances_mensuelles["date_performance"] <= fin_export)
].copy()

ponderations_export = ponderations_etape3.loc[
    (ponderations_etape3["date"] >= debut_export)
    & (ponderations_etape3["date"] <= fin_export)
].copy()
ponderations_long_export = ponderations_export.loc[ponderations_export["side"] == "long"].copy()
ponderations_short_export = ponderations_export.loc[ponderations_export["side"] == "short"].copy()

performances_export.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_performances.csv", index=False)
ponderations_export.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_ponderations.csv", index=False)
ponderations_long_export.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_portefeuille_long.csv", index=False)
ponderations_short_export.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_portefeuille_short.csv", index=False)
evaluation_resume.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_evaluation_lucky.csv", index=False)
stats_strategie.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_stats_strategie.csv", index=False)

resume_export_final = pd.DataFrame(
    {
        "fichier": [
            f"{PREFIXE_EXPORT_FINAL}_performances.csv",
            f"{PREFIXE_EXPORT_FINAL}_ponderations.csv",
            f"{PREFIXE_EXPORT_FINAL}_portefeuille_long.csv",
            f"{PREFIXE_EXPORT_FINAL}_portefeuille_short.csv",
            f"{PREFIXE_EXPORT_FINAL}_evaluation_lucky.csv",
            f"{PREFIXE_EXPORT_FINAL}_stats_strategie.csv",
        ],
        "description": [
            "Performances mensuelles de la strategie",
            "Ponderations detaillees long et short",
            "Ponderations du portefeuille long",
            "Ponderations du portefeuille short",
            "Comparaison strategie vs lucky",
            "Statistiques resumee de la strategie",
        ],
    }
)
resume_export_final.to_csv(DOSSIER_DONNEES / f"{PREFIXE_EXPORT_FINAL}_index.csv", index=False)

print("Exports CSV generes dans :", DOSSIER_DONNEES.resolve())
print(f"Periode cible d'export : {debut_export.date()} -> {fin_export.date()}")
print(f"Nombre de lignes exportees pour les performances : {len(performances_export)}")
print(f"Nombre de lignes exportees pour les ponderations : {len(ponderations_export)}")

if performances_export.empty:
    print("Aucune performance n'est disponible sur cette fenetre avec les donnees actuelles.")

display(performances_export.head())
display(ponderations_export.head(20))

Exports CSV generes dans : C:\00_Projets_VSC\Python\Projets_Derivatives\02_Strategie_long_short\02_Strategie_long_short\data
Periode cible d'export : 2008-03-01 -> 2023-03-31
Nombre de lignes exportees pour les performances : 2
Nombre de lignes exportees pour les ponderations : 90


,date_formation,date_performance,performance_long,performance_short,performance_strategie
0,2023-01-31,2023-02-28,0.041367,0.008645,0.050012
1,2023-02-28,2023-03-31,-0.041083,-0.087645,-0.128728


,date,ticker,company_name,score_global,rang,side,score_absolu,poids_absolu,poids_signe
0,2023-01-31,VOW3.DE,Volkswagen,1.695210,1,long,1.695210,0.198138,0.198138
1,2023-01-31,TTE.PA,TotalEnergies,1.004950,2,long,1.004950,0.117459,0.117459
2,2023-01-31,SAN.MC,Banco Santander,0.926817,3,long,0.926817,0.108327,0.108327
3,2023-01-31,ENGI.PA,Engie,0.914747,4,long,0.914747,0.106916,0.106916
4,2023-01-31,BNP.PA,BNP Paribas,0.909819,5,long,0.909819,0.106340,0.106340
5,2023-01-31,INGA.AS,ING,0.704319,6,long,0.704319,0.082321,0.082321
6,2023-01-31,BAYN.DE,Bayer,0.618802,7,long,0.618802,0.072326,0.072326
7,2023-01-31,MUV2.DE,Munich Re,0.463467,8,long,0.463467,0.054170,0.054170
8,2023-01-31,DTE.DE,Deutsche Telekom,0.411115,9,long,0.411115,0.048051,0.048051
9,2023-01-31,CS.PA,AXA,0.390064,10,long,0.390064,0.045591,0.045591


### Résultat du rendu final

Le notebook produit désormais une famille de fichiers CSV `rendu_final_*.csv` dans le dossier `data`.

On y retrouve :

- les performances mensuelles de la stratégie ;
- les pondérations mensuelles détaillées ;
- les portefeuilles long et short ;
- la comparaison avec les stratégies lucky ;
- les statistiques résumées de la stratégie ;
- un fichier `rendu_final_index.csv` qui récapitule le contenu des exports.

Si la fenêtre mars 2008 à mars 2023 reste partiellement ou totalement vide, cela signifie que la profondeur historique des données fondamentales disponibles n'est pas suffisante pour reconstruire le signal value sur toute la période visée.